# TFB (Training-Free Bayesianization) Example

This notebook demonstrates how to use TFB for uncertainty estimation with LoRA-finetuned models.

TFB converts pre-trained LoRA weights into Bayesian posteriors via SVD-based variance inference,
enabling stochastic sampling without additional training.

**Reference**: Shi et al. "Training-Free Bayesianization for Low-Rank Adapters of Large Language Models" (arXiv:2412.05723)

## Requirements
- TFB **requires trained LoRA weights** (non-zero B matrix). Freshly initialized LoRA won't produce diverse samples.
- For best results, use a LoRA adapter that was fine-tuned on your task.

In [11]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model
from datasets import load_dataset

from lm_polygraph import WhiteboxModel
from lm_polygraph.utils.tfb import (
    apply_tfb, 
    enable_tfb_sampling, 
    disable_tfb_sampling,
    fit_tfb_beta,
    _extract_lora_layers,
)
from lm_polygraph.estimators.tfb import TFBPredictiveEntropy, TFBSampleVariance
from lm_polygraph.stat_calculators.tfb_sample import TFBSamplingCalculator
from lm_polygraph.utils.generation_parameters import GenerationParameters

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load a Model with Trained LoRA

For meaningful TFB results, you need a model with **trained** (non-zero) LoRA-B weights.
Options:
1. Use a pre-trained LoRA from HuggingFace Hub
2. Train your own LoRA adapter
3. For demo: Initialize with small random values (shows mechanism, not realistic uncertainty)

In [17]:
# Small model for demo - works on Kaggle/Colab free tier

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = PeftModel.from_pretrained(model, "ShahzebKhoso/qwen2.5-instruct-0.5B-pubmedqa-lora")
tokenizer = AutoTokenizer.from_pretrained("ShahzebKhoso/qwen2.5-instruct-0.5B-pubmedqa-lora")

# CRITICAL: Move model to device BEFORE applying TFB
model = model.to(device)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded on {device}")

Using device: cpu
Model loaded on cpu


In [18]:

model.print_trainable_parameters()

trainable params: 0 || all params: 495,114,112 || trainable%: 0.0000


## Load PubMedQA Dataset

Load the PubMedQA dataset that the model was fine-tuned on. We'll use the `pqa_labeled` subset which contains questions requiring yes/no/maybe answers based on medical abstracts.

In [13]:
# Load PubMedQA dataset
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

# Take a subset for calibration and testing
calibration_samples = dataset.select(range(0, 250))  # 250 samples for calibration
test_samples = dataset.select(range(260, 270))  # 10 samples for testing

print(f"Loaded {len(calibration_samples)} calibration samples")
print(f"Loaded {len(test_samples)} test samples")

# Show example structure
print("\nExample question:")
example = test_samples[0]
print(f"Question: {example['question']}")
# print(f"Context: {example['context'][:200]}...")
print(f"Expected answer: {example['final_decision']}")

Loaded 250 calibration samples
Loaded 10 test samples

Example question:
Question: Does Residency Selection Criteria Predict Performance in Orthopaedic Surgery Residency?
Expected answer: yes


## Apply TFB and Verify Setup

TFB performs SVD on each LoRA-B matrix and computes variance parameters for noise injection.

In [21]:
# Apply TFB - this modifies LoRA layers in-place
BETA = 0.01  # Noise scale. Higher = more diverse samples
lora_layers = apply_tfb(model, beta=BETA)
print(f"Applied TFB to {len(lora_layers)} LoRA layers")

# Verify TFB attributes were added
layer = lora_layers[0]
print(f"\nLayer has TFB attributes:")
print(f"  - tfb_sampling_enabled: {layer.tfb_sampling_enabled}")
print(f"  - tfb_beta: {layer.tfb_beta}")
print(f"  - lora_A_rho shape: {layer.lora_A_rho['default'].shape}")

Applied TFB to 48 LoRA layers

Layer has TFB attributes:
  - tfb_sampling_enabled: False
  - tfb_beta: 0.01
  - lora_A_rho shape: torch.Size([16, 896])


## Verify Stochastic Behavior

When TFB sampling is enabled, the same input should produce different outputs.

In [ ]:
# Format a PubMedQA question for the model
def format_pubmed_question(sample):
    """Format PubMedQA sample using Qwen's chat template."""
    context = sample['context']
    question = sample['question']
    
    # Use Qwen's chat template format
    messages = [
        {
            "role": "system",
            "content": "You are a medical expert. Answer the question with 'yes', 'no', or 'maybe' based on the provided context."
        },
        {
            "role": "user",
            "content": f"Context: {context}\n\nQuestion: {question}\n\nAnswer with 'yes', 'no', or 'maybe':"
        }
    ]
    
    # Apply chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

test_input = format_pubmed_question(test_samples[0])
inputs = tokenizer(test_input, return_tensors="pt", truncation=True, max_length=512).to(device)

model.eval()

# Deterministic (sampling disabled)
disable_tfb_sampling(model)
with torch.no_grad():
    out1 = model(**inputs).logits
    out2 = model(**inputs).logits
print(f"Deterministic: outputs identical? {torch.allclose(out1, out2)}")

# Stochastic (sampling enabled)
enable_tfb_sampling(model)
outputs = []
with torch.no_grad():
    for i in range(5):
        out = model(**inputs).logits
        outputs.append(out.clone())

# Check variance across samples
stacked = torch.stack(outputs)
variance = stacked.var(dim=0).mean().item()
print(f"Stochastic: mean variance across samples: {variance:.6f}")
print(f"Stochastic: all same? {all(torch.allclose(outputs[0], o) for o in outputs[1:])}")

if variance < 1e-6:
    print("\n⚠️ WARNING: Low variance! Check that LoRA-B has non-zero weights.")
else:
    print(f"\n✓ TFB sampling working correctly (variance = {variance:.6f})")

Deterministic: outputs identical? True
Stochastic: mean variance across samples: 9.945613
Stochastic: all same? False

✓ TFB sampling working correctly (variance = 9.945613)


## Generate Samples with TFB

Use `TFBSamplingCalculator` to generate multiple samples from the posterior.

In [ ]:
# Wrap model for LM-Polygraph
gen_params = GenerationParameters(do_sample=False, max_new_tokens=50)
lm_model = WhiteboxModel(model, tokenizer, generation_parameters=gen_params)

# Generate TFB samples for a PubMedQA question
calculator = TFBSamplingCalculator(n_samples=20, beta=0.01)
enable_tfb_sampling(model)

sample = test_samples[0]
question = format_pubmed_question(sample)
stats = calculator({}, [question], lm_model, max_new_tokens=50)

print(f"Question: {sample['question']}")
print(f"Expected: {sample['final_decision']}")
print(f"\nTFB Samples:")
for i, (text, lp) in enumerate(zip(stats['tfb_sample_texts'][0], stats['tfb_sample_log_probs'][0])):
    print(f"  {i+1}. (log_prob={lp:.2f}) {text.strip()}")

Question: Does Residency Selection Criteria Predict Performance in Orthopaedic Surgery Residency?
Expected: yes

TFB Samples:
  1. (log_prob=-22.26) The orthopaedic residency selection criteria examined in the orthopaedic residency study are not the same as the orthopaedic orthopaedic orthopaedic orthopaedic orthopaedic orthopaedic orthopaedic orthopaedic orthopaedic orthopa
  2. (log_prob=-27.76) Yes, but the criteria used in the selection process are subjective and differ in terms of which criteria predict either objective measures of resident performance by faculty. The criteria used in the selection process are subjective and differ in terms of which criteria predict resident performance. The
  3. (log_prob=-15.50) The criteria used in the selection process often are subjective and differ in terms of which criteria predict either objective measures or subjective ratings of resident performance by faculty. The criteria used in the selection process is often subjective and differ in 

In [ ]:
# Compute uncertainty estimates
entropy_est = TFBPredictiveEntropy()
variance_est = TFBSampleVariance()

entropy = entropy_est(stats)[0]
variance = variance_est(stats)[0]

print(f"Uncertainty metrics:")
print(f"  Predictive Entropy: {entropy:.4f}")
print(f"  Sample Variance: {variance:.4f}")

## Compare Uncertainty Across Different PubMedQA Questions

Analyze uncertainty estimates on various medical questions. Questions with ambiguous or complex medical scenarios should show higher uncertainty.

In [ ]:
# Analyze uncertainty across multiple PubMedQA samples
print(f"{'Question':<80} {'Expected':<10} {'Entropy':<12} {'Variance':<12}")
print("-" * 115)

for sample in test_samples[:8]:
    question = format_pubmed_question(sample)
    stats = calculator({}, [question], lm_model, max_new_tokens=50)
    entropy = entropy_est(stats)[0]
    variance = variance_est(stats)[0]
    
    # Truncate question for display
    q_display = sample['question'][:75] + "..." if len(sample['question']) > 75 else sample['question']
    expected = sample['final_decision']
    
    print(f"{q_display:<80} {expected:<10} {entropy:<12.4f} {variance:<12.4f}")

## Optional: Calibrate Beta

The `fit_tfb_beta` function finds optimal beta that maximizes sample diversity
while keeping prediction degradation below a threshold.

In [ ]:
# Prepare calibration data from PubMedQA
calibration_texts = [
    format_pubmed_question(sample) 
    for sample in calibration_samples[:5]  # Use 5 samples for quick calibration
]
calibration_inputs = [
    tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    for text in calibration_texts
]

print(f"Using {len(calibration_inputs)} PubMedQA samples for calibration")

# Find optimal beta (quick calibration for demo)
optimal_beta = fit_tfb_beta(
    model,
    calibration_inputs,
    target_metric_ratio=0.01,  # 1% degradation target
    max_iters=5,
    n_samples=3,
    initial_beta=0.2,
    verbose=True,
)
print(f"\nOptimal beta: {optimal_beta:.6f}")

## Summary

**TFB workflow demonstrated on PubMedQA:**
1. Load model with **trained** LoRA adapter (this model is fine-tuned on PubMedQA)
2. `apply_tfb(model, beta)` - transforms LoRA for Bayesian sampling via SVD
3. `enable_tfb_sampling(model)` - activates stochastic forward passes
4. Use `TFBSamplingCalculator` to generate posterior samples
5. Use `TFBPredictiveEntropy` / `TFBSampleVariance` for uncertainty scores
6. Calibrate beta with `fit_tfb_beta` using domain-specific data

**Key parameters:**
- `beta`: Noise scale (0.01-0.3 typical). Higher = more diverse samples
- `n_samples`: Number of posterior samples (5-20 typical)

**Expected behavior:**
- Medical questions with clear yes/no answers should show **lower uncertainty**
- Ambiguous or complex medical scenarios should show **higher uncertainty**
- The uncertainty estimates can help identify when the model is less confident in its predictions